# LRTP Protocol Evaluation — Figure Generation
**CS3102 Practical 2** · pc1-036-l ↔ pc2-002-l · 22 Apr 2026

This notebook parses all test log files and produces publication-ready figures for the evaluation chapter.
All figures are saved to `figures/` as high-resolution PNGs and as PDFs for direct inclusion in LaTeX.

In [ ]:
import re
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter

os.makedirs('figures', exist_ok=True)

# ── Consistent style ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'legend.fontsize':  10,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'grid.linestyle':   '--',
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

# Colour palette
C_BLUE   = '#2563EB'
C_ORANGE = '#EA580C'
C_GREEN  = '#16A34A'
C_PURPLE = '#7C3AED'
C_GREY   = '#6B7280'
C_RED    = '#DC2626'

def save(name):
    plt.savefig(f'figures/{name}.png')
    plt.savefig(f'figures/{name}.pdf')
    plt.show()
    print(f'  → saved figures/{name}.{{png,pdf}}')

print('Setup complete.')

---
## 1 — Hard-coded test data
All values are extracted directly from the provided log files.

In [ ]:
# Read test metadata and per-test logs instead of hard-coded values
import glob
import json
from pathlib import Path

LOG_DIR = Path('exemplar_logs')
TEST_SUMMARY = LOG_DIR / 'test_summary.txt'

def parse_test_summary(path):
    tests = []
    client_map = {}
    if not path.exists():
        return tests, client_map
    with open(path) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        m = re.match(r'^Test \d+:\s+(.+?)\s+-\s+(PASSED|FAILED)\s+\((\d+)s\)', line)
        if m:
            name = m.group(1).strip()
            if name.isdigit():
                display_name = f'test-{name}'
            else:
                display_name = name
            passed = (m.group(2) == 'PASSED')
            duration_s = int(m.group(3))
            label = display_name.replace('test-', 'Test ').replace('-', ' ').title()
            tests.append({'name': display_name, 'label': label, 'duration_s': duration_s, 'passed': passed})
        # capture client log filename on following lines
        cm = re.search(r'Client Log:\s*(\S+_client\.log)', line)
        if cm:
            for j in range(max(0, i-3), i+1):
                m2 = re.match(r'^Test \d+:\s+(.+?)\s+-', lines[j]) if j < len(lines) else None
                if m2:
                    nm = m2.group(1).strip()
                    nm = f'test-{nm}' if nm.isdigit() else nm
                    client_map[nm] = LOG_DIR / cm.group(1)
                    break
    return tests, client_map

def parse_log_kv(path):
    d = {}
    text = path.read_text() if path.exists() else ''
    for m in re.finditer(r'^(\w[\w_]+)\s*:\s*(.+)$', text, flags=re.M):
        key, val = m.group(1), m.group(2).strip()
        if re.match(r'^-?\d+$', val.replace('_','')):
            try: d[key] = int(val.replace('_',''))
            except: d[key] = val
        elif re.match(r'^-?\d+\.\d+$', val):
            try: d[key] = float(val)
            except: d[key] = val
        else:
            d[key] = val
    m = re.search(r'Smoothed RTT:\s*(\d+)\s*us', text)
    if m: d['srtt_us'] = int(m.group(1))
    m = re.search(r'RTT Variance:\s*(\d+)\s*us', text)
    if m: d['rttvar_us'] = int(m.group(1))
    m = re.search(r'Final RTO:\s*(\d+)\s*us', text)
    if m: d['rto_us'] = int(m.group(1))
    m = re.search(r'\(duration\):\s*([0-9.]+)s', text)
    if m: d['duration'] = float(m.group(1))
    m = re.search(r'data mean tx_rate: .* -\s*(\d+)\s*bps', text)
    if m: d['tx_rate_bps'] = int(m.group(1))
    m = re.search(r'data mean rx_rate: .* -\s*(\d+)\s*bps', text)
    if m: d['rx_rate_bps'] = int(m.group(1))
    return d

def parse_packet_table(path):
    text = path.read_text() if path.exists() else ''
    lines = text.splitlines()
    start = None
    for i, ln in enumerate(lines):
        if re.match(r'\s*Packet\s*\|', ln):
            start = i+2
            break
    if start is None:
        return None
    pkt, rtt, srtt, rttvar, rto = [], [], [], [], []
    for ln in lines[start:]:
        if not ln.strip(): break
        parts = [p.strip() for p in ln.split('|')]
5